# Function 4: Dashboard Handoff & KPI Verification Starter

Welcome! All data cleaning, exploratory analysis, and machine learning propensity scoring are complete.
Use the datasets inside `dashboard_ready_data/` and follow `DASHBOARD_GUIDE.md` to build the 4-page dashboard (Power BI, Tableau, or Streamlit).

### Included Ready-to-Use Tables in `dashboard_ready_data/`:
1. **`customer_propensity_scores.csv`**: `52,013` eligible customers with `predicted_repeat_probability`, `propensity_percentile`, and `propensity_segment` (`High (top 5%)`, `Medium (next 15%)`, `Low (bottom 80%)`).
2. **`model_metrics.csv` & `top_k_lift_metrics.csv`**: Held-out test evaluation metrics (`PR-AUC`, `ROC-AUC`, `Top-K Lift`) for the Model Quality page.
3. **Cleaned Core Tables (`orders_clean.csv`, `customers_clean.csv`, `order_items_clean.csv`, `order_payments_clean.csv`, `order_reviews_clean.csv`, `products_clean.csv`, `category_translation_clean.csv`)**: For historical revenue, AOV, category, review, and regional visuals.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("dashboard_ready_data")

scores = pd.read_csv(DATA_DIR / "customer_propensity_scores.csv")
model_metrics = pd.read_csv(DATA_DIR / "model_metrics.csv")
lift_metrics = pd.read_csv(DATA_DIR / "top_k_lift_metrics.csv")
orders = pd.read_csv(DATA_DIR / "orders_clean.csv")
items = pd.read_csv(DATA_DIR / "order_items_clean.csv")
customers = pd.read_csv(DATA_DIR / "customers_clean.csv")

print(f"Propensity scores loaded: {scores.shape[0]:,} customers")
display(scores.head())
print("\nSegment Breakdown:")
display(scores.groupby("propensity_segment", as_index=False).agg(
    customer_count=("customer_unique_id", "count"),
    avg_probability=("predicted_repeat_probability", "mean")
))
print("\nModel Quality Metrics:")
display(model_metrics)
print("\nTop-K Lift Metrics:")
display(lift_metrics)


In [2]:
# Executive Overview KPI Reference Numbers (for Dashboard Validation)
items["item_revenue"] = items["price"] + items["freight_value"]
order_rev = items.groupby("order_id", as_index=False)["item_revenue"].sum()
delivered = orders[orders["order_status"] == "delivered"].merge(order_rev, on="order_id", how="left")

total_revenue = delivered["item_revenue"].sum()
total_orders = delivered["order_id"].nunique()
aov = total_revenue / total_orders

print(f"Total Delivered Revenue : R${total_revenue:,.2f}")
print(f"Total Delivered Orders  : {total_orders:,}")
print(f"Average Order Value     : R${aov:,.2f}")
